---

# MULTIPLE COLUMN CREATION WITH `.assign()`

---

The `.assign()` Method **creates multiple columns** at once and returns a DataFrame

    - This can be chained together with other data processing methods

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('Data/retail_2016_2017.csv')
df

,id,date,store_nbr,family,sales,onpromotion
0,1945944,2016-01-01,1,AUTOMOTIVE,0.000,0
1,1945945,2016-01-01,1,BABY CARE,0.000,0
2,1945946,2016-01-01,1,BEAUTY,0.000,0
3,1945947,2016-01-01,1,BEVERAGES,0.000,0
4,1945948,2016-01-01,1,BOOKS,0.000,0
...,...,...,...,...,...,...
1054939,3000883,2017-08-15,9,POULTRY,438.133,0
1054940,3000884,2017-08-15,9,PREPARED FOODS,154.553,1
1054941,3000885,2017-08-15,9,PRODUCE,2419.729,148
1054942,3000886,2017-08-15,9,SCHOOL AND OFFICE SUPPLIES,121.000,8


In [3]:
sample_df = df.sample(5, random_state=2023)
sample_df

,id,date,store_nbr,family,sales,onpromotion
613709,2559653,2016-12-10,29,DAIRY,731.0,15
290251,2236195,2016-06-11,52,HOME AND KITCHEN II,0.0,0
670069,2616013,2017-01-12,10,BOOKS,0.0,0
870367,2816311,2017-05-04,3,PERSONAL CARE,404.0,5
654702,2600646,2017-01-03,29,HOME AND KITCHEN I,30.0,7


---

In [8]:
sample_df = sample_df.assign(tax_amount=sample_df.sales * 0.05)
sample_df

,id,date,store_nbr,family,sales,onpromotion,tax_amount
613709,2559653,2016-12-10,29,DAIRY,731.0,15,36.55
290251,2236195,2016-06-11,52,HOME AND KITCHEN II,0.0,0,0.00
670069,2616013,2017-01-12,10,BOOKS,0.0,0,0.00
870367,2816311,2017-05-04,3,PERSONAL CARE,404.0,5,20.20
654702,2600646,2017-01-03,29,HOME AND KITCHEN I,30.0,7,1.50


In [10]:
sample_df = sample_df.assign(
    tax_amount = (sample_df.sales * 0.05).round(2).map(lambda x: f'${x}'),
    on_promotion_flag = sample_df.onpromotion > 0,
    year = sample_df.date.str[:4].astype('int')
).query('year < 2017')
sample_df

,id,date,store_nbr,family,sales,onpromotion,tax_amount,on_promotion_flag,year
613709,2559653,2016-12-10,29,DAIRY,731.0,15,$36.55,True,2016
290251,2236195,2016-06-11,52,HOME AND KITCHEN II,0.0,0,$0.0,False,2016


---

In [19]:
retail = pd.read_csv('Data/retail_2016_2017.csv')
sample_retail = retail.sample(5, random_state=85)
sample_retail

,id,date,store_nbr,family,sales,onpromotion
383220,2329164,2016-08-03,11,MEATS,852.20404,0
883307,2829251,2017-05-11,42,PREPARED FOODS,53.28200,2
656822,2602766,2017-01-04,38,MAGAZINES,4.00000,0
389404,2335348,2016-08-06,35,BOOKS,0.00000,0
216577,2162521,2016-05-01,35,SCHOOL AND OFFICE SUPPLIES,49.00000,7


In [20]:
sample_retail = sample_retail.assign(
    onpromotion_flag = sample_retail.onpromotion > 0,
    family_abbrev = sample_retail.family.str[:3],
    onpromotion_ratio = sample_retail.sales / sample_retail.onpromotion,
    sales_onprom_target = lambda x : x.onpromotion_ratio > 100
)    
sample_retail

,id,date,store_nbr,family,sales,onpromotion,onpromotion_flag,family_abbrev,onpromotion_ratio,sales_onprom_target
383220,2329164,2016-08-03,11,MEATS,852.20404,0,False,MEA,inf,True
883307,2829251,2017-05-11,42,PREPARED FOODS,53.28200,2,True,PRE,26.641,False
656822,2602766,2017-01-04,38,MAGAZINES,4.00000,0,False,MAG,inf,True
389404,2335348,2016-08-06,35,BOOKS,0.00000,0,False,BOO,NaN,False
216577,2162521,2016-05-01,35,SCHOOL AND OFFICE SUPPLIES,49.00000,7,True,SCH,7.000,False


In [21]:
sample_retail.query('sales_onprom_target == True')

,id,date,store_nbr,family,sales,onpromotion,onpromotion_flag,family_abbrev,onpromotion_ratio,sales_onprom_target
383220,2329164,2016-08-03,11,MEATS,852.20404,0,False,MEA,inf,True
656822,2602766,2017-01-04,38,MAGAZINES,4.00000,0,False,MAG,inf,True


---

# ASSIGNMENT: ASSIGN

---

In [26]:
transactions = pd.read_csv('Data/transactions.csv')
transactions.columns = ['date', 'store_number', 'transactions_count']
transactions

,date,store_number,transactions_count
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922
...,...,...,...
83483,2017-08-15,50,2804
83484,2017-08-15,51,1573
83485,2017-08-15,52,2255
83486,2017-08-15,53,932


In [35]:
conditions = [
    (transactions['date'].astype('datetime64[ns]').dt.month == 12),
    (transactions['date'].astype('datetime64[ns]').dt.month == 5) & (transactions['date'].astype('datetime64[ns]').dt.dayofweek == 6),
    (transactions['date'].astype('datetime64[ns]').dt.month == 7) & (transactions['date'].astype('datetime64[ns]').dt.day_of_week == 0)
]

choices = ['Holiday Bonus', 'Corporate Month', 'Summer Special']

# ---------------------------------------------------------------------------------------
transactions = transactions.assign(
    target_pct = transactions.transactions_count / 2500,
    met_target = transactions.target_pct >=1,
    bonus_payable = transactions.met_target * 100,
    month = transactions.date.astype('datetime64[ns]').dt.month,
    day_of_week = transactions.date.astype('datetime64[ns]').dt.dayofweek,
    seasonal_bonus = np.select(condlist=conditions, choicelist=choices, default='None')
)

transactions

,date,store_number,transactions_count,target_pct,met_target,bonus_payable,month,day_of_week,seasonal_bonus
0,2013-01-01,25,770,0.3080,False,0,1,1,None
1,2013-01-02,1,2111,0.8444,False,0,1,2,None
2,2013-01-02,2,2358,0.9432,False,0,1,2,None
3,2013-01-02,3,3487,1.3948,True,100,1,2,None
4,2013-01-02,4,1922,0.7688,False,0,1,2,None
...,...,...,...,...,...,...,...,...,...
83483,2017-08-15,50,2804,1.1216,True,100,8,1,None
83484,2017-08-15,51,1573,0.6292,False,0,8,1,None
83485,2017-08-15,52,2255,0.9020,False,0,8,1,None
83486,2017-08-15,53,932,0.3728,False,0,8,1,None


In [36]:
transactions.seasonal_bonus.value_counts()

seasonal_bonus
None               75259
Holiday Bonus       6028
Summer Special      1103
Corporate Month     1098
Name: count, dtype: int64

In [41]:
transactions[transactions.seasonal_bonus != 'None']['transactions_count'].count() * 100

np.int64(822900)